# WDC Schema.org Class Analysis: ClaimReview & Event

**Source:** [Web Data Commons 2024-12 — Schema.org Class-Specific Subsets](https://webdatacommons.org/structureddata/2024-12/stats/schema_org_subsets.html)

## Strategy

**ClaimReview** (56 MB, ~3.9M quads, ~49.7K URLs)  
→ Downloaded directly from the WDC class-specific subset: a single `part_0.gz` file.  
→ Data is **merged JSON-LD + Microdata** — cannot be separated within this subset.

**Event** (~20.83 GB, ~1.96B quads, ~14.1M URLs, 133 parts)  
→ Downloaded from the WDC class-specific subset: 133 `part_N.gz` files.  
→ Data is **merged JSON-LD + Microdata** — cannot be separated within this subset.

## Analyses

1. **Syntactic / Structural Validity** — namespace correctness, type validity, property validity, domain/range violations  
2. **Semantic Completeness & Richness** — prevalence, property coverage, AvP, characteristic sets

In [ ]:
# !pip install requests pandas lxml tqdm

In [1]:
import requests, gzip, io, re, json, time, threading
from collections import defaultdict, Counter
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

# ── WDC endpoints ──────────────────────────────────────────────────────────────
WDC_CLASS_BASE = "https://data.dws.informatik.uni-mannheim.de/structureddata/2024-12/quads/classspecific"

# ── Schema.org namespaces ──────────────────────────────────────────────────────
SO_HTTP  = "http://schema.org/"
SO_HTTPS = "https://schema.org/"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

# ── Local cache paths ──────────────────────────────────────────────────────────
DATA_DIR      = Path("wdc_class_data")
DATA_DIR.mkdir(exist_ok=True)

CLAIMREVIEW_NQ  = DATA_DIR / "ClaimReview.nq.gz"
EVENT_PARTS_DIR = DATA_DIR / "Event"
EVENT_PARTS_DIR.mkdir(exist_ok=True)

print("Setup complete.")
print(f"  ClaimReview  → {CLAIMREVIEW_NQ}")
print(f"  Event parts  → {EVENT_PARTS_DIR}/")

Setup complete.
  ClaimReview  → wdc_class_data/ClaimReview.nq.gz
  Event parts  → wdc_class_data/Event/


## Part 1 — ClaimReview: Download Class-Specific Subset

WDC provides a ready-made `ClaimReview/part_0.gz` (~56 MB compressed) containing all quads whose source pages include at least one `schema:ClaimReview` entity. This covers the full corpus — not filtered by domain.

> **Note on format separation:** The class-specific subsets merge JSON-LD and Microdata into a single file. There is no flag per quad to distinguish the extractor. We accept this limitation here.

In [2]:
def download_class_subset(class_name: str, dest: Path, part: int = 0):
    """Download a WDC class-specific subset part file (keeps it gzipped locally)."""
    url = f"{WDC_CLASS_BASE}/{class_name}/part_{part}.gz"
    if dest.exists():
        print(f"  Already cached: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"  Downloading {url} …")
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        done  = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:.1f} / {total/1e6:.1f} MB", end="", flush=True)
    print(f"\n  Saved → {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

download_class_subset("ClaimReview", CLAIMREVIEW_NQ)

  Already cached: wdc_class_data/ClaimReview.nq.gz (57.6 MB)


In [3]:
# ── N-Quads parser ─────────────────────────────────────────────────────────────
NQ_RE = re.compile(
    r'^'
    r'(<[^>]*>|_:\S+)'          # subject
    r'\s+'
    r'(<[^>]+>)'                 # predicate (always URI)
    r'\s+'
    r'(<[^>]*>|_:\S+|"(?:[^"\\]|\\.)*"(?:@\w[\w-]*|(?:\^\^<[^>]+>))?)'  # object
    r'\s+'
    r'(<[^>]+>)'                 # graph (named graph = source page URL)
    r'\s+\.\s*$'
)

def parse_nq_line(line: str):
    """Return (subject, predicate, object, graph) strings or None."""
    m = NQ_RE.match(line)
    if not m:
        return None
    return m.group(1), m.group(2), m.group(3), m.group(4)

def strip_angle(s: str) -> str:
    return s[1:-1] if s.startswith("<") and s.endswith(">") else s


# ── Build entity store ─────────────────────────────────────────────────────────
# entity_store[subject] = {
#   "types":  set of schema.org class URIs this entity declares itself to be
#   "props":  {predicate_URI: {object_str: set of graph URIs}}
#   "graphs": set of all source page URIs this entity appeared in
# }

def build_entity_store(nq_path: Path, gz: bool = True, capture_errors: bool = False):
    """
    Parse an nquads file and build a per-subject entity store with
    triple-level provenance: each (predicate, object) pair records
    the set of source URLs (graphs) it was found in.

    Returns:
        entity_store : dict as described above
        parse_errors : count of unparseable lines
        total_lines  : total lines read
        error_lines  : list of unparseable lines (only populated if capture_errors=True)
    """
    def make_entity():
        return {
            "types":  set(),
            "props":  defaultdict(lambda: defaultdict(set)),  # pred → obj → {graphs}
            "graphs": set(),
        }

    entity_store = defaultdict(make_entity)
    parse_errors = 0
    total        = 0
    error_lines  = []

    opener = (
        gzip.open(nq_path, "rt", encoding="utf-8", errors="replace")
        if gz else
        open(nq_path, encoding="utf-8", errors="replace")
    )

    with opener as f:
        for line in f:
            total += 1
            parsed = parse_nq_line(line.rstrip("\n"))
            if not parsed:
                parse_errors += 1
                if capture_errors:
                    error_lines.append(line.strip())
                continue

            subj, pred, obj, graph = parsed
            graph_uri = strip_angle(graph)
            es = entity_store[subj]
            es["graphs"].add(graph_uri)

            pred_uri = strip_angle(pred)
            if pred_uri == RDF_TYPE:
                es["types"].add(strip_angle(obj))
            else:
                es["props"][pred_uri][obj].add(graph_uri)  # triple → source URLs

    return dict(entity_store), parse_errors, total, error_lines


print("Building ClaimReview entity store …")
cr_store, cr_errors, cr_total, bad_lines = build_entity_store(CLAIMREVIEW_NQ, gz=True, capture_errors=True)
print(f"  Lines parsed : {cr_total:,}")
print(f"  Parse errors : {cr_errors:,}")
print(f"  Entities     : {len(cr_store):,}")

Building ClaimReview entity store …
  Lines parsed : 3,919,703
  Parse errors : 2
  Entities     : 787,354


In [4]:
for line in bad_lines:
    print(repr(line))


'_:node1ifgc31blx3524023 <https://schema.org/url</div>> <https://www.politifact.com/new-hampshire/statements/2017/apr/02/blog-posting/will-federal-government-require-devices-slow-down-/> <https://api.politifact.com/factchecks/2017/apr/02/blog-posting/will-federal-government-require-devices-slow-down-/>   .'
'_:node1ifencembx41827023 <https://schema.org/url</div>> <https://www.politifact.com/new-hampshire/statements/2017/apr/02/blog-posting/will-federal-government-require-devices-slow-down-/> <https://www.politifact.com/factchecks/2017/apr/02/blog-posting/will-federal-government-require-devices-slow-down-/>   .'


## Part 2 — Load Schema.org Vocabulary

We fetch the full schema.org vocabulary (JSON-LD format) to:
- Enumerate all valid type names
- Map each type to its valid properties (via `domainIncludes`)
- Map each property to its expected value types (via `rangeIncludes`)

In [5]:
VOCAB_CACHE = DATA_DIR / "schemaorg_vocab.json"

def load_schemaorg_vocab():
    if VOCAB_CACHE.exists():
        print("Loading vocab from cache …")
        with open(VOCAB_CACHE) as f:
            raw = json.load(f)
    else:
        url = "https://schema.org/version/latest/schemaorg-current-https.jsonld"
        print(f"Fetching schema.org vocab from {url} …")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw = r.json()
        with open(VOCAB_CACHE, "w") as f:
            json.dump(raw, f)
        print(f"  Cached → {VOCAB_CACHE}")

    graph = raw.get("@graph", [])
    print(f"  {len(graph):,} vocabulary items")

    def local(uri: str) -> str:
        # Handles both full URIs (https://schema.org/ClaimReview)
        # and compact forms (schema:ClaimReview)
        return uri.split("/")[-1].split("#")[-1].split(":")[-1]

    def ids_of(val) -> list[str]:
        if isinstance(val, dict):
            return [val.get("@id", "")]
        if isinstance(val, list):
            return [v.get("@id", "") if isinstance(v, dict) else str(v) for v in val]
        return [str(val)]

    all_types    = set()
    type_props   = defaultdict(set)
    prop_range   = defaultdict(set)
    type_parents = defaultdict(set)

    for item in graph:
        item_id   = item.get("@id", "")
        item_type = item.get("@type", "")
        if "rdfs:Class" in str(item_type) or "schema:DataType" in str(item_type):
            name = local(item_id)
            all_types.add(name)
            for parent_id in ids_of(item.get("rdfs:subClassOf", [])):
                if parent_id:
                    type_parents[name].add(local(parent_id))

    for item in graph:
        item_id   = item.get("@id", "")
        item_type = item.get("@type", "")
        if "rdf:Property" in str(item_type):
            prop_name = local(item_id)
            for domain_id in ids_of(item.get("schema:domainIncludes", [])):
                if domain_id:
                    type_props[local(domain_id)].add(prop_name)
            for range_id in ids_of(item.get("schema:rangeIncludes", [])):
                if range_id:
                    prop_range[prop_name].add(local(range_id))

    def get_ancestors(t, memo={}):
        if t in memo:
            return memo[t]
        ancestors = set()
        for parent in type_parents.get(t, set()):
            ancestors.add(parent)
            ancestors |= get_ancestors(parent, memo)
        memo[t] = ancestors
        return ancestors

    for t in list(all_types):
        for ancestor in get_ancestors(t):
            type_props[t] |= type_props.get(ancestor, set())

    print(f"  Classes : {len(all_types):,}")
    print(f"  With properties defined: {sum(1 for v in type_props.values() if v):,}")
    return all_types, dict(type_props), dict(prop_range), dict(type_parents)


so_types, so_type_props, so_prop_range, so_type_parents = load_schemaorg_vocab()

for cls in ["ClaimReview", "NewsArticle", "Article"]:
    props = so_type_props.get(cls, set())
    print(f"\n  {cls}: {len(props)} properties defined")

Loading vocab from cache …
  3,219 vocabulary items
  Classes : 979
  With properties defined: 918

  ClaimReview: 139 properties defined

  NewsArticle: 141 properties defined

  Article: 136 properties defined


3,219 vocabulary items: total entries in the schema.org including classes, properties, enumerations, and data types combined.

Classes: 979: how many of those 3,219 items are classes (e.g. ClaimReview, Person, Organization).

With properties defined: 918: how many of those 979 classes have at least one property directly defined on them via domainIncludes. The remaining 61 are abstract/leaf classes with no properties of their own (they only inherit).

In [6]:
# ── Helper: filter entity store to a target class ─────────────────────────────
def filter_by_class(store: dict, target_class: str) -> dict:
    """Return only entities that have target_class (or a subclass) among their types."""
    result = {}
    for subj, data in store.items():
        for t in data["types"]:
            local_t = t.split("/")[-1].split("#")[-1]
            if local_t == target_class or target_class in so_type_parents.get(local_t, set()):
                result[subj] = data
                break
    return result

cr_entities = filter_by_class(cr_store, "ClaimReview")

print(f"ClaimReview entities : {len(cr_entities):,}")

ClaimReview entities : 60,313


Example Entites after parsing

In [7]:
for i, (subj, data) in enumerate(list(cr_entities.items())[:3]):
    print(f"\n{'═'*70}")
    print(f"[{i+1}] {subj}")
    print(f"  types  : {data['types']}")
    print(f"  graphs : {data['graphs']}")
    print(f"  props:")
    for pred, obj_graphs in data["props"].items():
        prop_name = pred.split("/")[-1]
        for obj, graphs in obj_graphs.items():
            print(f"    {prop_name:<25} = {obj[:60]}")
            print(f"    {'':25}   ↳ {graphs}")



══════════════════════════════════════════════════════════════════════
[1] _:nc7e1182ed5a447b0a414f588fc50c392xb0
  types  : {'http://schema.org/ClaimReview'}
  graphs : {'https://checagem.ig.com.br/2023-02-09/envie-a-url-da-materia-e-conferimos-para-voce.html'}
  props:
    author                    = _:nc7e1182ed5a447b0a414f588fc50c392xb1
                                ↳ {'https://checagem.ig.com.br/2023-02-09/envie-a-url-da-materia-e-conferimos-para-voce.html'}
    claimReviewed             = "Ficou em dúvida sobre a veracidade de um conteúdo? Envie pa
                                ↳ {'https://checagem.ig.com.br/2023-02-09/envie-a-url-da-materia-e-conferimos-para-voce.html'}
    itemReviewed              = _:nc7e1182ed5a447b0a414f588fc50c392xb2
                                ↳ {'https://checagem.ig.com.br/2023-02-09/envie-a-url-da-materia-e-conferimos-para-voce.html'}
    reviewRating              = _:nc7e1182ed5a447b0a414f588fc50c392xb9
                                ↳ {'http

In [8]:
# Examples of entities with multiple source URLs
multi_source = [(s, d) for s, d in cr_entities.items() if len(d["graphs"]) > 1]
print(f"Entities with multiple source URLs: {len(multi_source):,}\n")

for i, (subj, data) in enumerate(multi_source[:3]):
    print(f"\n{'═'*70}")
    print(f"[{i+1}] {subj}")
    print(f"  types  : {data['types']}")
    print(f"  graphs : {data['graphs']}")
    print(f"  props:")
    for pred, obj_graphs in data["props"].items():
        prop_name = pred.split("/")[-1]
        for obj, graphs in obj_graphs.items():
            print(f"    {prop_name:<25} = {obj[:60]}")
            print(f"    {'':25}   ↳ {graphs}")


Entities with multiple source URLs: 41


══════════════════════════════════════════════════════════════════════
[1] <https://api.fullfact.org/content/claim-reviews/4f0edc3f-0e3a-41c9-ab5d-388edfa4c7cf/>
  types  : {'http://schema.org/ClaimReview'}
  graphs : {'https://fullfact.org/crime/knife-crime-since-2010/', 'https://fullfact.org/crime/knife-crime-since-2010/?utm_source=content_page&utm_medium=related_content'}
  props:
    author                    = _:n8fc72c45ee884838b3d32cb6a4183909xb6
                                ↳ {'https://fullfact.org/crime/knife-crime-since-2010/'}
    author                    = _:n1dface51b9ce44159753c8c71622d6fexb6
                                ↳ {'https://fullfact.org/crime/knife-crime-since-2010/?utm_source=content_page&utm_medium=related_content'}
    claimReviewed             = "Knife crime is up by 70% since 2015."
                                ↳ {'https://fullfact.org/crime/knife-crime-since-2010/', 'https://fullfact.org/crime/knife-crime-s

## Part 4 — Syntactic / Structural Validity

### 4.1 Namespace Correctness Rate

Check that schema.org types and properties are referenced with the canonical namespace (`http://schema.org/` or `https://schema.org/`).  
Flag: wrong capitalisation, trailing slashes, typos, or use of other namespaces.

In [9]:
CANONICAL_NS = (SO_HTTP, SO_HTTPS)

def namespace_report(entities: dict, label: str):
    canonical = wrong = other = 0
    wrong_examples = Counter()

    for subj, data in entities.items():
        for t in data["types"]:
            if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS):
                canonical += 1
            elif re.search(r'schema\.(org|org/)', t, re.I):
                wrong += 1
                wrong_examples[t] += 1
            else:
                other += 1

        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS):
                canonical += 1
            elif re.search(r'schema\.(org|org/)', prop, re.I):
                wrong += 1
                wrong_examples[prop] += 1
            else:
                other += 1

    total_schema = canonical + wrong
    rate = canonical / total_schema * 100 if total_schema else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Canonical schema.org URIs : {canonical:>8,}")
    print(f"  Malformed schema.org URIs : {wrong:>8,}")
    print(f"  Other namespace URIs      : {other:>8,}  (rdf:, owl:, etc.)")
    print(f"  Namespace correctness rate: {rate:.2f}%")
    if wrong_examples:
        print(f"  Malformed examples (top 5):")
        for uri, cnt in wrong_examples.most_common(5):
            print(f"    [{cnt:>5}]  {uri}")
    return rate

cr_ns_rate = namespace_report(cr_entities, "ClaimReview")


────────────────────────────────────────────────────────────
  ClaimReview
────────────────────────────────────────────────────────────
  Canonical schema.org URIs :  316,261
  Malformed schema.org URIs :        0
  Other namespace URIs      :        0  (rdf:, owl:, etc.)
  Namespace correctness rate: 100.00%


60,313 entities
× ~5 properties each   → ~300,000 predicate URI occurrences

### 4.2 Type Validity Rate

For every `rdf:type` assertion that references schema.org, check whether the local name is a known class in the schema.org vocabulary.

Catches: wrong capitalisation (`newsarticle` vs `NewsArticle`), invented types, deprecated terms.

In [10]:
def type_validity_report(entities: dict, label: str):
    valid = Counter()
    invalid = Counter()

    for subj, data in entities.items():
        for t in data["types"]:
            if not (t.startswith(SO_HTTP) or t.startswith(SO_HTTPS)):
                continue
            local_t = t.split("/")[-1].split("#")[-1]
            if local_t in so_types:
                valid[local_t] += 1
            else:
                invalid[local_t] += 1

    total = sum(valid.values()) + sum(invalid.values())
    rate  = sum(valid.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid types   : {sum(valid.values()):>8,}  ({rate:.2f}%)")
    print(f"  Invalid types : {sum(invalid.values()):>8,}  ({100-rate:.2f}%)")
    print(f"\n  Valid type distribution (top 10):")
    for t, c in valid.most_common(10):
        print(f"    {t:<45} {c:>8,}")
    if invalid:
        print(f"\n  Invalid types (top 10):")
        for t, c in invalid.most_common(10):
            print(f"    {t:<45} {c:>8,}")
    return rate, valid, invalid

cr_type_rate, cr_valid_types, cr_invalid_types = type_validity_report(cr_entities, "ClaimReview")


────────────────────────────────────────────────────────────
  ClaimReview
────────────────────────────────────────────────────────────
  Valid types   :   60,539  (100.00%)
  Invalid types :        1  (0.00%)

  Valid type distribution (top 10):
    ClaimReview                                     60,313
    Review                                             225
    ReportageNewsArticle                                 1

  Invalid types (top 10):
    ExplainerNewsArticle                                 1


### 4.3 Property Validity Rate

For each entity, check whether its schema.org properties are actually defined for its type (via `domainIncludes`, with inheritance).

A property is **valid** if it appears in the schema.org vocabulary and `domainIncludes` the entity's type or any of its ancestors.

In [11]:
def get_all_types_for_entity(data: dict) -> set[str]:
    result = set()
    for t in data["types"]:
        if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS):
            result.add(t.split("/")[-1].split("#")[-1])
    return result

def prop_validity_report(entities: dict, label: str):
    valid_props   = Counter()
    invalid_props = Counter()

    for subj, data in entities.items():
        entity_types = get_all_types_for_entity(data)
        allowed = set()
        for et in entity_types:
            allowed |= so_type_props.get(et, set())

        for prop in data["props"]:
            if not (prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)):
                continue
            prop_local = prop.split("/")[-1].split("#")[-1]
            if prop_local in allowed:
                valid_props[prop_local] += 1
            else:
                invalid_props[prop_local] += 1

    total = sum(valid_props.values()) + sum(invalid_props.values())
    rate  = sum(valid_props.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid property uses   : {sum(valid_props.values()):>8,}  ({rate:.2f}%)")
    print(f"  Invalid property uses : {sum(invalid_props.values()):>8,}  ({100-rate:.2f}%)")
    print(f"\n  Most used valid properties (top 15):")
    for p, c in valid_props.most_common(15):
        print(f"    {p:<45} {c:>8,}")
    if invalid_props:
        print(f"\n  Invalid properties (top 10):")
        for p, c in invalid_props.most_common(10):
            print(f"    {p:<45} {c:>8,}")
    return rate, valid_props, invalid_props

cr_prop_rate, cr_valid_props, cr_invalid_props = prop_validity_report(cr_entities, "ClaimReview")


────────────────────────────────────────────────────────────
  ClaimReview
────────────────────────────────────────────────────────────
  Valid property uses   :  255,057  (99.74%)
  Invalid property uses :      664  (0.26%)

  Most used valid properties (top 15):
    url                                             36,466
    author                                          36,274
    claimReviewed                                   36,020
    reviewRating                                    35,766
    itemReviewed                                    33,730
    datePublished                                   32,992
    dateModified                                     6,813
    image                                            6,306
    publisher                                        5,649
    mainEntityOfPage                                 5,278
    name                                             5,049
    description                                      4,949
    keywords              

### 4.4 Domain/Range Violation Rate

Check whether object values match the expected range for their property.

- **Object properties** (range = a schema.org class): value should be a URI (`<…>`) or blank node, not a string literal
- **Date/time properties** (`datePublished`, `dateModified`, etc.): value should match ISO 8601 pattern
- **URL properties** (`url`, `sameAs`): value should be a URI, not a plain literal

In [ ]:
import re
from collections import Counter, defaultdict

SO_HTTP  = "http://schema.org/"
SO_HTTPS = "https://schema.org/"

DATE_PROPS = {
    "datePublished", "dateModified", "dateCreated", "expires",
    "startDate", "endDate", "uploadDate"
}

URL_PROPS = {"url", "sameAs", "mainEntityOfPage", "image", "thumbnail"}

LITERAL_DATATYPES = {
    "Text", "Number", "Integer", "Float", "Boolean",
    "Date", "DateTime", "Time", "URL", "Duration"
}

# --- Strict ISO 8601 (with milliseconds) ---
ISO_DATE_RE = re.compile(
    r'^\d{4}'
    r'(?:-\d{2}'
    r'(?:-\d{2}'
    r'(?:T\d{2}:\d{2}'
    r'(?::\d{2}(?:\.\d+)?)?'
    r'(?:Z|[+-]\d{2}:\d{2})?'
    r')?)?)?$'
)

LANG_RE = re.compile(r"^[a-zA-Z]{2,3}(-[a-zA-Z0-9]{2,8})*$")

def is_literal(obj: str) -> bool:
    return obj.startswith('"')

def is_uri(obj: str) -> bool:
    return obj.startswith("<")

def is_blank(obj: str) -> bool:
    return obj.startswith("_:")

def strip_literal(obj: str) -> str:
    m = re.match(r'^"((?:[^"\\]|\\.)*)"', obj)
    return m.group(1) if m else obj

def normalize_date(val: str) -> str:
    if " " in val and "T" not in val:
        val = val.replace(" ", "T")
    return val

# --- Literal validation by datatype ---

def validate_literal(val: str, ranges: set, prop: str) -> bool:
    val = val.strip()

    if "Text" in ranges:
        return True  # any string is valid text

    if "URL" in ranges:
        return val.startswith("http")

    if "Boolean" in ranges:
        return val in {"true", "false", "0", "1"}

    if "Integer" in ranges:
        return re.fullmatch(r"-?\d+", val) is not None

    if "Float" in ranges or "Number" in ranges:
        return re.fullmatch(r"-?\d+(\.\d+)?", val) is not None

    if "Date" in ranges or "DateTime" in ranges:
        return ISO_DATE_RE.match(normalize_date(val)) is not None

    if prop == "inLanguage":
        return LANG_RE.match(val) is not None

    return False


def range_violation_report_strict(entities: dict, label: str, so_prop_range: dict):
    violations = Counter()
    ok = Counter()
    viol_examples = defaultdict(list)

    for subj, data in entities.items():
        for prop, values in data["props"].items():

            if not (prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)):
                continue

            prop_local = prop.split("/")[-1].split("#")[-1]
            ranges = so_prop_range.get(prop_local, set())

            literal_ranges = ranges & LITERAL_DATATYPES
            entity_ranges  = ranges - LITERAL_DATATYPES

            for obj in values:
                key = prop_local

                valid = False

                # --- LITERAL ---
                if is_literal(obj):
                    val = strip_literal(obj)

                    if literal_ranges:
                        if validate_literal(val, literal_ranges, prop_local):
                            valid = True

                # --- ENTITY ---
                elif is_uri(obj) or is_blank(obj):
                    if entity_ranges:
                        # NOTE: minimal check here (see below for upgrade)
                        valid = True

                # --- RESULT ---
                if valid:
                    ok[key] += 1
                else:
                    violations[key] += 1
                    if len(viol_examples[key]) < 3:
                        if is_literal(obj):
                            viol_examples[key].append(f"lit={strip_literal(obj)[:60]!r}")
                        else:
                            viol_examples[key].append(f"uri={obj[:60]!r}")

    total = sum(violations.values()) + sum(ok.values())
    viol_rate = (sum(violations.values()) / total * 100) if total else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label} (STRICT)")
    print(f"{'─'*60}")
    print(f"  Checked value assertions : {total:>8,}")
    print(f"  Range violations         : {sum(violations.values()):>8,}  ({viol_rate:.2f}%)")

    print(f"\n  Top violation properties (top 10):")
    for p, c in violations.most_common(10):
        exs = viol_examples.get(p, [])
        print(f"    {p:<35} {c:>6,}  e.g. {exs}")

    return viol_rate, violations

cr_range_rate, cr_violations = range_violation_report(
    cr_entities,
    "ClaimReview",
    so_prop_range  
)


────────────────────────────────────────────────────────────
  ClaimReview
────────────────────────────────────────────────────────────
  Checked value assertions :  258,093
  Range violations         :    6,201  (2.40%)

  Top violation properties (top 10):
    datePublished                        5,043  e.g. ["date='10/08/2023'", "date='3 Nov. 2018'", "date='21 Aug. 2020'"]
    potentialAction                        488  e.g. ["lit='SKF Authorized Distributor'", "lit='SKF Authorized Distributor'", "lit='SKF Authorized Distributor'"]
    thumbnailUrl                           304  e.g. ["uri='<https://www.placemania.sk/wp-content/uploads/2018/10/palmam'", "uri='<https://www.placemania.sk/wp-content/uploads/2020/01/nyugat'", "uri='<https://www.placemania.sk/wp-content/uploads/2018/01/letisk'"]
    article                                270  e.g. ["uri='_:n96ae681e62e0433e969c0f0985645f3bxb0'", "uri='_:na29765d1b4db4e77bb6824a69955411fxb0'", "uri='_:n5915ff1ef6f244f1afc8aa9d6b75142axb0

data issues:

- `datePublished` → Non-ISO formats (`"10/08/2023"`, `"3 Nov. 2018"`)  
  Must follow ISO 8601 (`YYYY-MM-DD` or `YYYY-MM-DDTHH:MM:SSZ`)

- `potentialAction` → String instead of `Action` object  
  Requires a structured entity (e.g. `{"@type": "ViewAction", ...}`)

- `thumbnailUrl` → IRI node instead of string URL  
  Schema.org (especially JSON-LD) expects literal URLs

- `article`, `claimant` → Entity without verified `@type`  
  Cannot confirm it matches expected classes (`CreativeWork`, `Person`, etc.)

- `reviewRating` → Literal instead of `Rating` object  
  Must be structured (`{"@type": "Rating", "ratingValue": ...}`)

- `sameAs` → Empty string  
  Must be a valid URL

- `author` → Plain string instead of `Person` or `Organization`  
  Strict validation requires structured entities

- `itemReviwed` → Invalid property (typo)  
  Should be `itemReviewed`

- `image` → Invalid value (`"\\n\\n\\n"`)  
  Must be a URL or `ImageObject`



## Part 5 — Semantic Completeness & Richness

### 5.1 Prevalence & Domain Distribution

How many entities exist per class, and how are they distributed across domains?

In [18]:
def get_pld(url: str) -> str:
    try:
        host = urlparse(url).netloc or url
        parts = host.split(".")
        return ".".join(parts[-2:]) if len(parts) >= 2 else host
    except Exception:
        return url

def prevalence_report(entities: dict, label: str, top_n: int = 20):
    domain_counts = Counter()
    for subj, data in entities.items():
        # Count each domain once per entity (deduplicated across graphs)
        seen = set()
        for graph in data.get("graphs", set()) or {subj}:
            d = get_pld(graph)
            if d not in seen:
                seen.add(d)
                domain_counts[d] += 1

    total = sum(domain_counts.values())
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Total entities : {total:,}")
    print(f"  Unique domains : {len(domain_counts):,}")
    print(f"\n  Top {top_n} domains by entity count:")
    df = pd.DataFrame(domain_counts.most_common(top_n), columns=["domain", "count"])
    df["% of total"] = (df["count"] / total * 100).round(2)
    print(df.to_string(index=False))
    return domain_counts

cr_domains = prevalence_report(cr_entities, "ClaimReview")


────────────────────────────────────────────────────────────
  ClaimReview
────────────────────────────────────────────────────────────
  Total entities : 60,313
  Unique domains : 318

  Top 20 domains by entity count:
            domain  count  % of total
        vpravda.ru  22891       37.95
           afp.com   2918        4.84
             co.th   2337        3.87
       boomlive.in   2152        3.57
   leadstories.com   1720        2.85
            org.pl   1667        2.76
     healthday.com   1330        2.21
   provereno.media   1255        2.08
    types-cars.com   1157        1.92
   vishvasnews.com   1000        1.66
        misbar.com    963        1.60
            com.br    894        1.48
      fullfact.org    855        1.42
    politifact.com    787        1.30
        maldita.es    769        1.28
          tempo.co    629        1.04
truthorfiction.com    609        1.01
     correctiv.org    577        0.96
      newsmeter.in    567        0.94
        facta.news 

vpravda.ru is an online newspaper focused on the Volgograd region in Russia

afp is global french news agency (like Reuters or AP)

### 5.2 Property Coverage

Of all properties *officially defined* for a class in schema.org, what fraction are actually used by at least one entity?

This measures breadth of adoption across the vocabulary — a coverage of 20% means only 1 in 5 defined properties ever appear in the wild.

In [20]:
def coverage_report(entities: dict, target_class: str):
    defined_props = so_type_props.get(target_class, set())
    if not defined_props:
        print(f"  No properties defined in vocab for {target_class}")
        return

    used_props = set()
    for data in entities.values():
        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS):
                used_props.add(prop.split("/")[-1].split("#")[-1])

    used_and_defined = used_props & defined_props
    defined_not_used = defined_props - used_props
    coverage = len(used_and_defined) / len(defined_props) * 100

    print(f"\n{'─'*60}")
    print(f"  {target_class} — Property Coverage")
    print(f"{'─'*60}")
    print(f"  Properties defined in schema.org : {len(defined_props):>5}")
    print(f"  Properties used (at least once)  : {len(used_and_defined):>5}  ({coverage:.1f}% coverage)")
    print(f"  Properties never used            : {len(defined_not_used):>5}  ({100-coverage:.1f}% gap)")

    prop_freq = Counter()
    for data in entities.values():
        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS):
                p = prop.split("/")[-1].split("#")[-1]
                if p in defined_props:
                    prop_freq[p] += 1
    n = len(entities) or 1

    print(f"\n  Used defined properties (freq = # entities using it, desc):")
    rows = [(p, prop_freq[p], prop_freq[p]/n*100) for p in defined_props]
    rows.sort(key=lambda x: -x[1])
    df = pd.DataFrame(rows, columns=["property", "entities_using", "density_%"])
    df["density_%"] = df["density_%"].round(2)
    print(df[df["entities_using"] > 0].to_string(index=False))
    print(f"\n  Never-used properties: {sorted(defined_not_used)}")

coverage_report(cr_entities, "ClaimReview")


────────────────────────────────────────────────────────────
  ClaimReview — Property Coverage
────────────────────────────────────────────────────────────
  Properties defined in schema.org :   139
  Properties used (at least once)  :    41  (29.5% coverage)
  Properties never used            :    98  (70.5% gap)

  Used defined properties (freq = # entities using it, desc):
            property  entities_using  density_%
                 url           36466      60.46
              author           36274      60.14
       claimReviewed           36020      59.72
        reviewRating           35766      59.30
        itemReviewed           33730      55.92
       datePublished           32992      54.70
        dateModified            6813      11.30
               image            6306      10.46
           publisher            5649       9.37
    mainEntityOfPage            5278       8.75
                name            5049       8.37
         description            4949       8

### 5.3 AvP — Average Properties per Entity

AvP (Average number of filled Properties per entity) measures annotation richness. Reported raw and as a fraction of the total available properties for the class.

In [21]:
def avp_report(entities: dict, target_class: str):
    defined_props = so_type_props.get(target_class, set())
    n_defined = len(defined_props) or 1

    prop_counts = []
    for data in entities.values():
        so_props_used = {
            prop.split("/")[-1].split("#")[-1]
            for prop in data["props"]
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        }
        prop_counts.append(len(so_props_used))

    if not prop_counts:
        print(f"  No entities for {target_class}")
        return

    import statistics
    avp = statistics.mean(prop_counts)
    med = statistics.median(prop_counts)
    std = statistics.stdev(prop_counts) if len(prop_counts) > 1 else 0
    mn  = min(prop_counts)
    mx  = max(prop_counts)

    print(f"\n{'─'*60}")
    print(f"  {target_class} — Average Properties per Entity (AvP)")
    print(f"{'─'*60}")
    print(f"  Entities            : {len(prop_counts):>8,}")
    print(f"  Available props     : {n_defined:>8,}  (schema.org defined)")
    print(f"  AvP (mean)          : {avp:>8.2f}  ({avp/n_defined*100:.1f}% of available)")
    print(f"  Median              : {med:>8.1f}")
    print(f"  Std dev             : {std:>8.2f}")
    print(f"  Min / Max           : {mn:>4}  /  {mx:>4}")

    buckets = Counter()
    for c in prop_counts:
        buckets[min(c, 20)] += 1

    print(f"\n  Distribution (# distinct SO properties per entity):")
    for k in range(0, min(mx + 2, 22)):
        bar_len = buckets.get(k, 0) * 40 // max(buckets.values(), default=1)
        label = f"{k:>2}" if k < 20 else "20+"
        print(f"    {label}: {'█' * bar_len} {buckets.get(k,0):,}")

avp_report(cr_entities, "ClaimReview")


────────────────────────────────────────────────────────────
  ClaimReview — Average Properties per Entity (AvP)
────────────────────────────────────────────────────────────
  Entities            :   60,313
  Available props     :      139  (schema.org defined)
  AvP (mean)          :     4.24  (3.1% of available)
  Median              :      6.0
  Std dev             :     3.79
  Min / Max           :    0  /    24

  Distribution (# distinct SO properties per entity):
     0: ████████████████████████████████████████ 22,896
     1:  165
     2:  14
     3:  161
     4: █ 976
     5: █████████ 5,587
     6: ██████████████████████████████ 17,220
     7: █████████ 5,474
     8: ██ 1,524
     9:  492
    10: ███████ 4,359
    11:  475
    12:  18
    13:  31
    14:  0
    15: █ 633
    16:  15
    17:  1
    18:  3
    19:  0
    20+:  269
    20+:  0


### 5.4 Characteristic Sets — Most Common Property Combinations

A *characteristic set* is the exact set of properties an entity uses. Finding the most frequent sets reveals the de-facto annotation patterns publishers actually follow, independent of what schema.org defines.

In [22]:
def characteristic_sets_report(entities: dict, label: str, top_n: int = 15):
    set_counter = Counter()

    for data in entities.values():
        so_props = frozenset(
            prop.split("/")[-1].split("#")[-1]
            for prop in data["props"]
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        )
        set_counter[so_props] += 1

    total = len(entities) or 1
    print(f"\n{'─'*60}")
    print(f"  {label} — Characteristic Sets (top {top_n})")
    print(f"{'─'*60}")
    print(f"  Unique property sets : {len(set_counter):,}")
    print()
    for i, (prop_set, count) in enumerate(set_counter.most_common(top_n), 1):
        pct = count / total * 100
        props_str = ", ".join(sorted(prop_set)) if prop_set else "(no SO properties)"
        print(f"  #{i:>2}  [{count:>6,}  {pct:>5.1f}%]  {{{props_str}}}")
    return set_counter

cr_char_sets = characteristic_sets_report(cr_entities, "ClaimReview")


────────────────────────────────────────────────────────────
  ClaimReview — Characteristic Sets (top 15)
────────────────────────────────────────────────────────────
  Unique property sets : 105

  # 1  [22,896   38.0%]  {(no SO properties)}
  # 2  [16,737   27.8%]  {author, claimReviewed, datePublished, itemReviewed, reviewRating, url}
  # 3  [ 3,885    6.4%]  {author, claimReviewed, dateModified, datePublished, image, itemReviewed, mainEntityOfPage, publisher, reviewRating, url}
  # 4  [ 3,335    5.5%]  {author, claimReviewed, itemReviewed, reviewRating, url}
  # 5  [ 2,329    3.9%]  {author, claimReviewed, datePublished, itemReviewed, name, reviewRating, url}
  # 6  [ 1,719    2.9%]  {author, claimReviewed, datePublished, description, itemReviewed, reviewRating, url}
  # 7  [ 1,091    1.8%]  {author, claimReviewed, datePublished, reviewRating, url}
  # 8  [   963    1.6%]  {author, claimReviewed, datePublished, description, itemReviewed, name, reviewRating, url}
  # 9  [   878    

In [24]:
empty_entities = {
    ent_id: data
    for ent_id, data in cr_entities.items()
    if not any(
        prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        for prop in data["props"]
    )
}

for ent_id, data in empty_entities.items():
    print("\n" + "=" * 80)
    print("ID:", ent_id)
    print("PROPS:")
    for p in data["props"]:
        print("  ", p)
    print("FULL RECORD:")
    print(data)


ID: _:node1ifdncec2x8895543
PROPS:
FULL RECORD:
{'types': {'http://schema.org/ClaimReview'}, 'props': defaultdict(<function build_entity_store.<locals>.make_entity.<locals>.<lambda> at 0x28d7334c0>, {}), 'graphs': {'https://verafiles.org/articles/vera-files-fact-check-duterte-wrong-claim-martial-law-and-na'}}

ID: _:node1ifdfpiv3x16259307
PROPS:
FULL RECORD:
{'types': {'http://schema.org/ClaimReview'}, 'props': defaultdict(<function build_entity_store.<locals>.make_entity.<locals>.<lambda> at 0x28d7b0d60>, {}), 'graphs': {'https://verafiles.org/articles/vera-files-fact-check-joma-sison-did-not-say-he-wants-dutert'}}

ID: _:node1iffuthtix1577150
PROPS:
FULL RECORD:
{'types': {'http://schema.org/ClaimReview'}, 'props': defaultdict(<function build_entity_store.<locals>.make_entity.<locals>.<lambda> at 0x28d7b1940>, {}), 'graphs': {'https://verafiles.org/articles/vera-files-fact-check-media-groups-err-figures-media-killing'}}

ID: _:node1iffuthtix1577151
PROPS:
FULL RECORD:
{'types': {'ht

---

# WDC Schema.org Class Analysis: Event

**Source:** [Web Data Commons 2024-12 — Schema.org Class-Specific Subsets](https://webdatacommons.org/structureddata/2024-12/stats/schema_org_subsets.html)

## Strategy

**Event** — Downloaded directly from the WDC class-specific subset: a single `part_0.gz` file.  
→ Data is **merged JSON-LD + Microdata** — cannot be separated within this subset.

## Analyses

1. **Syntactic / Structural Validity** — namespace correctness, type validity, property validity, domain/range violations  
2. **Semantic Completeness & Richness** — prevalence, property coverage, AvP, characteristic sets

## Part 1 — Event: Download Class-Specific Subset

In [ ]:
def download_all_parts(class_name: str, dest_dir: Path) -> list[Path]:
    """
    Download all part_N.gz files for a WDC class-specific subset.
    Stops when the server returns 404. Returns list of local paths.
    """
    downloaded = []
    part = 0
    while True:
        url  = f"{WDC_CLASS_BASE}/{class_name}/part_{part}.gz"
        dest = dest_dir / f"part_{part}.gz"
        if dest.exists():
            print(f"  [part {part}] Already cached ({dest.stat().st_size / 1e6:.1f} MB)")
            downloaded.append(dest)
            part += 1
            continue
        r = requests.head(url, timeout=30)
        if r.status_code == 404:
            print(f"  [part {part}] 404 — no more parts. Total: {part} parts.")
            break
        r.raise_for_status()
        print(f"  [part {part}] Downloading {url} …")
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            total = int(r.headers.get("Content-Length", 0))
            done  = 0
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        print(f"\r    {done/1e6:.1f} / {total/1e6:.1f} MB", end="", flush=True)
        print(f"\n  [part {part}] Saved → {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
        downloaded.append(dest)
        part += 1
    return downloaded

event_parts = download_all_parts("Event", EVENT_PARTS_DIR)
total_size  = sum(p.stat().st_size for p in event_parts) / 1e9
print(f"\n  Event: {len(event_parts)} parts, {total_size:.2f} GB on disk")

In [ ]:
def build_entity_store_multipart(part_paths: list[Path], capture_errors: bool = False):
    """Build a single entity store by streaming over multiple nq.gz part files."""
    def make_entity():
        return {
            "types":  set(),
            "props":  defaultdict(lambda: defaultdict(set)),
            "graphs": set(),
        }

    entity_store = defaultdict(make_entity)
    total_lines  = 0
    total_errors = 0
    all_bad      = []

    for path in part_paths:
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
            for line in f:
                total_lines += 1
                parsed = parse_nq_line(line.rstrip("\n"))
                if not parsed:
                    total_errors += 1
                    if capture_errors:
                        all_bad.append(line.strip())
                    continue
                subj, pred, obj, graph = parsed
                graph_uri = strip_angle(graph)
                es = entity_store[subj]
                es["graphs"].add(graph_uri)
                pred_uri = strip_angle(pred)
                if pred_uri == RDF_TYPE:
                    es["types"].add(strip_angle(obj))
                else:
                    es["props"][pred_uri][obj].add(graph_uri)
        print(f"  Done: {path.name}  (running total: {total_lines:,} lines, {len(entity_store):,} entities)")

    return dict(entity_store), total_errors, total_lines, all_bad


print("Building Event entity store from all parts …")
ev_store, ev_errors, ev_total, ev_bad_lines = build_entity_store_multipart(event_parts, capture_errors=False)
print(f"\n  Lines parsed : {ev_total:,}")
print(f"  Parse errors : {ev_errors:,}")
print(f"  Entities     : {len(ev_store):,}")

In [ ]:
for line in ev_bad_lines[:5]:
    print(repr(line))

In [ ]:
ev_entities = filter_by_class(ev_store, "Event")
print(f"Event entities : {len(ev_entities):,}")

In [ ]:
for i, (subj, data) in enumerate(list(ev_entities.items())[:3]):
    print(f"\n{'═'*70}")
    print(f"[{i+1}] {subj}")
    print(f"  types  : {data['types']}")
    print(f"  graphs : {data['graphs']}")
    print(f"  props:")
    for pred, obj_graphs in data["props"].items():
        prop_name = pred.split("/")[-1]
        for obj, graphs in obj_graphs.items():
            print(f"    {prop_name:<25} = {obj[:60]}")
            print(f"    {'':25}   ↳ {graphs}")

## Part 4 — Event: Syntactic / Structural Validity

### 4.1 Namespace Correctness Rate

In [ ]:
ev_ns_rate = namespace_report(ev_entities, "Event")

### 4.2 Type Validity Rate

In [ ]:
ev_type_rate, ev_valid_types, ev_invalid_types = type_validity_report(ev_entities, "Event")

### 4.3 Property Validity Rate

In [ ]:
ev_prop_rate, ev_valid_props, ev_invalid_props = prop_validity_report(ev_entities, "Event")

### 4.4 Domain/Range Violation Rate

In [ ]:
ev_range_rate, ev_violations = range_violation_report(ev_entities, "Event")

## Part 5 — Event: Semantic Completeness & Richness

### 5.1 Prevalence & Domain Distribution

In [ ]:
ev_domains = prevalence_report(ev_entities, "Event")

### 5.2 Property Coverage

In [ ]:
coverage_report(ev_entities, "Event")

### 5.3 AvP — Average Properties per Entity

In [ ]:
avp_report(ev_entities, "Event")

### 5.4 Characteristic Sets — Most Common Property Combinations

In [ ]:
ev_char_sets = characteristic_sets_report(ev_entities, "Event")

## Summary — Event

| Metric | Event |
|--------|-------|
| Source | WDC class-specific subset (merged JSON-LD+MD) |
| Entities | — |
| Namespace correctness | — |
| Type validity rate | — |
| Property validity rate | — |
| Range violation rate | — |
| Property coverage | — |
| AvP | — |

*Fill in after running all cells above.*